<a href="https://colab.research.google.com/github/wieland-github/information_extraction_german_medical_data/blob/main/gbert_gptnermed_training_base_genutzt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline (Off the shelf Model) fine tune auf GPTNERMED
- Hier nutze ich deepset/gbert-base als baseline.
- Model wird auf dem Vorhandenen GPTNERMED fine getuned
- Ergebnisse über drei Seeds gemittelt

In [1]:
!pip install -q \
  "transformers>=4.40,<4.46" \
  "datasets>=2.16.0,<3.0.0" \
  "accelerate>=0.30" \
  "seqeval" \
  "evaluate>=0.4.0,<0.5.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 57.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3

In [2]:
import torch
import datasets
import transformers

print(torch.cuda.is_available())

True


## Daten laden

In [3]:
def get_label(label_id):
    label_mapping = {
        0: "Medikation",
        1: "Dosis",
        2: "Diagnose",
    }
    return label_mapping.get(label_id)

print(get_label(0))

Medikation


In [4]:
def get_classes(labels):
    return [get_label(c) for c in labels["ner_class"]]

def to_offset_format(example):
    labels = example["ner_labels"]  # dict mit Listen: start / stop / ner_class
    return {
        "text": example["sentence"],
        "ent_starts": labels["start"],
        "ent_stops": labels["stop"],
        "ent_classes": get_classes(labels),
    }

print(to_offset_format({"sentence": "Test", "ner_labels": {"start": [0], "stop": [2], "ner_class": [0]}}))

{'text': 'Test', 'ent_starts': [0], 'ent_stops': [2], 'ent_classes': ['Medikation']}


In [5]:
import datasets

raw = datasets.load_dataset("jfrei/GPTNERMED", trust_remote_code=True)
dataset = raw.map(to_offset_format, remove_columns=["sentence", "ner_labels"])

print(f"Loaded {len(dataset['train'])} train, {len(dataset['validation'])} dev and {len(dataset['test'])} test")
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/7876 [00:00<?, ? examples/s]

Map:   0%|          | 0/984 [00:00<?, ? examples/s]

Map:   0%|          | 0/985 [00:00<?, ? examples/s]

Loaded 7876 train, 984 dev and 985 test
{'text': '0,4 Diuretika 0,25 1x/die', 'ent_starts': [0, 4, 14, 19], 'ent_stops': [3, 13, 18, 25], 'ent_classes': ['Dosis', 'Medikation', 'Dosis', 'Dosis']}


## Daten vorverarbeiten

BIO schema:
- B: Anfang der Entität
- I: Innerhalb der Entität
- O: keine Entität


In [6]:
label_list = ["O", "B-Medikation", "I-Medikation", "B-Dosis", "I-Dosis", "B-Diagnose", "I-Diagnose"]
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

print(id2label)
print(label2id)

{0: 'O', 1: 'B-Medikation', 2: 'I-Medikation', 3: 'B-Dosis', 4: 'I-Dosis', 5: 'B-Diagnose', 6: 'I-Diagnose'}
{'O': 0, 'B-Medikation': 1, 'I-Medikation': 2, 'B-Dosis': 3, 'I-Dosis': 4, 'B-Diagnose': 5, 'I-Diagnose': 6}


In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("deepset/gbert-base")

tokenizer_config.json:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/362 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [8]:
def get_token_label(start, end, entities):
    if start == end:
        return -100

    for ent_start, ent_stop, ent_class in entities:
        if ent_start <= start and end <= ent_stop:
            prefix = "B-" if start == ent_start else "I-"
            return label2id[prefix + ent_class]

    return label2id["O"]

print(get_token_label(0, 4, [(0, 8, "Medikation")]))
print(get_token_label(3, 8, [(0, 8, "Medikation")]))

1
2


In [9]:
def get_entities(example):
    return list(zip(example["ent_starts"], example["ent_stops"], example["ent_classes"]))

print(get_entities(dataset["train"][0]))

[(0, 3, 'Dosis'), (4, 13, 'Medikation'), (14, 18, 'Dosis'), (19, 25, 'Dosis')]


In [10]:
def align_labels(offsets, entities):
    return [get_token_label(start, end, entities) for start, end in offsets]

enc = tokenizer("Aspirin 500mg", return_offsets_mapping=True)
print(align_labels(enc["offset_mapping"], [(0, 7, "Medikation"), (8, 13, "Dosis")]))


[-100, 1, 2, 2, 3, 4, -100]


In [11]:
def preprocess_function(example):
    tokenized = tokenizer(example["text"], truncation=True, max_length=128, return_offsets_mapping=True)
    entities = get_entities(example)
    tokenized["labels"] = align_labels(tokenized.pop("offset_mapping"), entities)
    return tokenized

print(preprocess_function(dataset["train"][0]))


{'input_ids': [102, 430, 818, 470, 2182, 1947, 23816, 430, 818, 1693, 164, 30950, 1061, 128, 103], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 3, 4, 4, 1, 2, 2, 3, 4, 4, 3, 4, 4, 4, -100]}


In [12]:
tokenized_dataset = dataset.map(
    preprocess_function,
    remove_columns=["text", "ent_starts", "ent_stops", "ent_classes"],
)
tokenized_dataset = tokenized_dataset.shuffle(seed=42)


Map:   0%|          | 0/7876 [00:00<?, ? examples/s]

Map:   0%|          | 0/984 [00:00<?, ? examples/s]

Map:   0%|          | 0/985 [00:00<?, ? examples/s]

In [13]:
example = tokenized_dataset["train"][0]
tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])
for token, label in zip(tokens, example["labels"]):
    print(token, "->", id2label[label] if label != -100 else "-100")

[CLS] -> -100
D -> O
: -> O
Die -> O
Wund -> O
##e -> O
wird -> O
mit -> O
1 -> B-Dosis
ml -> I-Dosis
Bu -> B-Medikation
##pi -> I-Medikation
##va -> I-Medikation
##ca -> I-Medikation
##in -> I-Medikation
versorgt -> O
. -> O
[SEP] -> -100


In [14]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [16]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []
    for pred, lab in zip(predictions, labels):
        sentence_preds = []
        sentence_labels = []
        for p, l in zip(pred, lab):
            if l != -100:
                sentence_preds.append(label_list[p])
                sentence_labels.append(label_list[l])
        true_predictions.append(sentence_preds)
        true_labels.append(sentence_labels)

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    metrics = {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

    for entity in ["Medikation", "Dosis", "Diagnose"]:
      if entity in results:
          metrics[f"{entity}_precision"] = results[entity]["precision"]
          metrics[f"{entity}_recall"] = results[entity]["recall"]
          metrics[f"{entity}_f1"] = results[entity]["f1"]

    return metrics

## Training (über drei Seeds)

In [17]:
import wandb
wandb.init(mode="disabled")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


In [18]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, set_seed

seeds = [42, 72, 102]
seed_results = []

for seed in seeds:
    print(f"\n===== Training mit Seed {seed} =====")
    set_seed(seed)

    model = AutoModelForTokenClassification.from_pretrained(
        "deepset/gbert-base", num_labels=7, id2label=id2label, label2id=label2id
    )

    training_args = TrainingArguments(
        output_dir=f"gbert_gptnermed_seed{seed}",
        evaluation_strategy="epoch",
        seed=seed,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # Test-Metriken für diesen Seed sichern
    test_metrics = trainer.predict(tokenized_dataset["test"]).metrics
    test_metrics["seed"] = seed
    seed_results.append(test_metrics)
    print(f"Seed {seed}: Test-F1 = {test_metrics['test_f1']:.4f}")


===== Training mit Seed 42 =====


model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at deepset/gbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Medikation Precision,Medikation Recall,Medikation F1,Dosis Precision,Dosis Recall,Dosis F1,Diagnose Precision,Diagnose Recall,Diagnose F1
1,0.288900,0.189848,0.847177,0.891387,0.868720,0.939784,0.927686,0.920082,0.923868,0.848718,0.888591,0.868197,0.739071,0.850629,0.790936
2,0.138900,0.187357,0.853414,0.901570,0.876831,0.941570,0.907000,0.929303,0.918016,0.875969,0.910067,0.892693,0.754190,0.849057,0.798817
3,0.071800,0.208516,0.879568,0.898600,0.888982,0.945040,0.927254,0.927254,0.927254,0.904891,0.893960,0.899392,0.785920,0.860063,0.821321


Seed 42: Test-F1 = 0.8981

===== Training mit Seed 72 =====


...

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Medikation Precision,Medikation Recall,Medikation F1,Dosis Precision,Dosis Recall,Dosis F1,Diagnose Precision,Diagnose Recall,Diagnose F1
1,0.279200,0.183937,0.856196,0.876538,0.866247,0.937742,0.910387,0.915984,0.913177,0.855670,0.891275,0.873110,0.775573,0.798742,0.786987
2,0.138900,0.183133,0.870378,0.888842,0.879513,0.945193,0.909910,0.931352,0.920506,0.893475,0.900671,0.897059,0.783866,0.809748,0.796597
3,0.075800,0.213311,0.885390,0.894782,0.890061,0.946775,0.927329,0.928279,0.927803,0.909847,0.880537,0.894952,0.799708,0.860063,0.828788


Seed 72: Test-F1 = 0.8911

===== Training mit Seed 102 =====


Some weights of BertForTokenClassification were not initialized from the model checkpoint at deepset/gbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Medikation Precision,Medikation Recall,Medikation F1,Dosis Precision,Dosis Recall,Dosis F1,Diagnose Precision,Diagnose Recall,Diagnose F1
1,0.279100,0.189916,0.846992,0.890115,0.868018,0.937181,0.896690,0.915984,0.906234,0.890521,0.895302,0.892905,0.734610,0.844340,0.785662
2,0.136000,0.186040,0.870583,0.899024,0.884575,0.943356,0.917004,0.928279,0.922607,0.913934,0.897987,0.905890,0.761905,0.855346,0.805926
3,0.073500,0.198161,0.886071,0.904115,0.895002,0.948459,0.924471,0.940574,0.932453,0.930652,0.900671,0.915416,0.784370,0.852201,0.816880


Seed 102: Test-F1 = 0.8953


## Ergebnisse

In [19]:
import numpy as np

print("Test-Ergebnisse pro Seed:")
for r in seed_results:
    print(f"  Seed {r['seed']}: F1={r['test_f1']:.4f}  Precision={r['test_precision']:.4f}  Recall={r['test_recall']:.4f}")

f1_scores = [r["test_f1"] for r in seed_results]
prec_scores = [r["test_precision"] for r in seed_results]
rec_scores = [r["test_recall"] for r in seed_results]

print("\nÜber alle Seeds gemittelt (Mittelwert ± Std):")
print(f"  F1:        {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"  Precision: {np.mean(prec_scores):.4f} ± {np.std(prec_scores):.4f}")
print(f"  Recall:    {np.mean(rec_scores):.4f} ± {np.std(rec_scores):.4f}")

Test-Ergebnisse pro Seed:
  Seed 42: F1=0.8981  Precision=0.8879  Recall=0.9085
  Seed 72: F1=0.8911  Precision=0.8802  Recall=0.9022
  Seed 102: F1=0.8953  Precision=0.8876  Recall=0.9030

Über alle Seeds gemittelt (Mittelwert ± Std):
  F1:        0.8948 ± 0.0029
  Precision: 0.8853 ± 0.0035
  Recall:    0.9045 ± 0.0028


In [33]:
categories = ["Diagnose", "Medikation", "Dosis"]

metrics = {
    "F1": "f1",
    "Precision": "precision",
    "Recall": "recall",
}

print("Verschiedene Kategorien über alle Seeds gemittelt (Mittelwert ± Std):")

for category in categories:
    print(f"\n{category}:")

    for metric_name, metric_key in metrics.items():
        scores = [
            r[f"test_{category}_{metric_key}"]
            for r in seed_results
        ]

        print(
            f"  {metric_name:10s}: "
            f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"
        )

Verschiedene Kategorien über alle Seeds gemittelt (Mittelwert ± Std):

Diagnose:
  F1        : 0.8126 ± 0.0075
  Precision : 0.7945 ± 0.0091
  Recall    : 0.8317 ± 0.0067

Medikation:
  F1        : 0.9380 ± 0.0017
  Precision : 0.9338 ± 0.0027
  Recall    : 0.9422 ± 0.0008

Dosis:
  F1        : 0.9043 ± 0.0032
  Precision : 0.8959 ± 0.0023
  Recall    : 0.9127 ± 0.0043
